The most part of this code was kindly provided by Korian Wicki, PhD Student in the
Sleep and Old Age research group at University Psychiatry Services Bern. Thank you!  
  
The code applies the YASA automatic sleep staging algorithm (Vallat & Walker, 2021) to pilot data from the current study.

In [14]:
# -------------------- CONFIGURE ME --------------------
FOLDER = "Pilotdaten/YASA_scored"  # ← put your folder here
EPOCH_SEC = 30           # epoch length (30s typical)
EXPORT_SET = True        # ← set to True to also export EEGLAB .set
# -----------------------------------------------------

#%pip install mne
#%pip install yasa
#%pip install edfio
#%pip install openpyxl
#%pip install eeglabio

import os, re, glob, shutil, mne, yasa, edfio, openpyxl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]



In [15]:
# Helper Functions - utiliy functions

#- `find_eeg_channel(ch_names)` → finds the EEG channel (FpZ required).
#- `pick_one(ch_names, candidates)` → picks the first available channel (EOG/EMG).
#- `already_scored(raw_path)` → checks if this EDF has already been scored (so we don’t process it twice).

def find_eeg_channel(ch_names):
    """
    Return a suitable EEG channel name.
    We prefer FpZ (any case variant). If not found, raise an error.
    """
    lower_map = {c.lower(): c for c in ch_names}
    for want in ["fpz", "fpz."]:  # add aliases here if your data uses different labels
        if want in lower_map:
            return lower_map[want]
    raise ValueError("No EEG channel 'FpZ' found in this recording.")


def pick_one(ch_names, candidates):
    """
    Return the first available channel from 'candidates' (or None).
    """
    for c in candidates:
        if c in ch_names:
            return c
    return None


def already_scored(raw_path):
    """
    Skip rules:
    - If input already ends with _scored.edf → skip.
    - If <basename>_scored.edf exists and is as new/newer than the raw → skip.
    """
    if raw_path.endswith("_scored.edf"):
        return True
    folder = os.path.dirname(raw_path)
    base = os.path.splitext(os.path.basename(raw_path))[0]
    base = re.sub(r'(_scored)+$', '', base)
    scored = os.path.join(folder, base + "_scored.edf")
    return os.path.exists(scored) and os.path.getmtime(scored) >= os.path.getmtime(raw_path)


In [16]:
def score_edf(edf_path):
    """
    Score a single EDF file in-place (non-recursive):
    - Skip if already scored
    - Run YASA sleep staging
    - Export EDF with annotations, hypnogram (PNG+CSV), metrics (Excel)
    - (Optional) Export EEGLAB .set if EXPORT_SET=True
    """
    if already_scored(edf_path):
        print(f"⏭️  Skipping (already scored & up-to-date): {os.path.basename(edf_path)}")
        return

    print(f"🔄 Processing: {os.path.basename(edf_path)}")

    # Load EDF
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    chs = raw.ch_names

    # Select channels
    eeg = find_eeg_channel(chs)             # required
    eog = pick_one(chs, ["EOG1", "EOG2"])   # optional
    emg = pick_one(chs, ["EMG1", "EMG2"])   # optional
    print(f"→ EEG: {eeg} | EOG: {eog} | EMG: {emg}")

    # Sleep staging
    sls = yasa.SleepStaging(raw, eeg_name=eeg, eog_name=eog, emg_name=emg)
    hypno = sls.predict()  # list of 'W','N1','N2','N3','R'
    hypno_int = hypno.as_int()     # numpy-Array mit 0,1,2,3,4 (W,N1,N2,N3,R)
    hypno_str = np.asarray(hypno.hypno)   # Strings: 'W','N1','N2','N3','R'

    # Add annotations into EDF
    annotations = mne.Annotations(
        onset=np.arange(0, len(hypno_str) * EPOCH_SEC, EPOCH_SEC),
        duration=[EPOCH_SEC] * len(hypno_str),
        description=hypno_str.tolist()
    )
    raw.set_annotations(annotations)

    # Build output filenames
    base = re.sub(r'(_scored)+$', '', os.path.splitext(os.path.basename(edf_path))[0])
    folder = os.path.dirname(edf_path)
    edf_out = os.path.join(folder, base + "_scored.edf")
    csv_out = os.path.join(folder, base + "_hypnogram.csv")
    png_out = os.path.join(folder, base + "_hypnogram.png")
    xls_out = os.path.join(folder, base + "_metrics.xlsx")
    set_out = os.path.join(folder, base + "_scored.set")

    # Save EDF with annotations
    raw.export(edf_out, fmt="edf", overwrite=True)

    # Plot hypnogram
    time_hours = np.arange(len(hypno)) * EPOCH_SEC / 3600
    plt.figure(figsize=(12, 4))
    plt.plot(time_hours, hypno_int, drawstyle="steps-post")
    plt.yticks([0, 1, 2, 3, 4], ["W", "N1", "N2", "N3", "REM"])
    plt.xlabel("Time (h)")
    plt.ylabel("Stage")
    plt.title(f"Hypnogram – {base}")
    plt.grid()
    plt.savefig(png_out, dpi=300)
    plt.close()

    # Save hypnogram (CSV) + summary metrics (Excel)
    pd.DataFrame({"Time (h)": time_hours, "Stage": hypno}).to_csv(csv_out, index=False)
    summary = yasa.sleep_statistics(hypno_int, sf_hyp=1/EPOCH_SEC)
    pd.DataFrame(summary, index=[0]).to_excel(xls_out, index=False)

    # Optional: export EEGLAB .set
    if EXPORT_SET:
        try:
            # mne>=1.3: export_raw is in mne.export
            from mne.export import export_raw
            export_raw(raw=raw, fname=set_out, fmt='eeglab', overwrite=True)
            print(f"💾 Exported EEGLAB .set: {set_out}")
        except Exception as e:
            print(f"⚠️ EEGLAB .set export failed: {e}")

    print(f"✅ Done: {os.path.basename(edf_out)}\n")


In [17]:
# Find EDF files directly inside the folder (not recursive)
edf_files = glob.glob(os.path.join(FOLDER, "*.edf"))

print(f"Found {len(edf_files)} EDF file(s) in {FOLDER}")

for f in edf_files:
    score_edf(f)

print("All done.")

Found 23 EDF file(s) in Pilotdaten/YASA_scored
⏭️  Skipping (already scored & up-to-date): 04-49-15-4B-D3-6E_Axo-Device-075_20250707_204037_000009.edf
⏭️  Skipping (already scored & up-to-date): 04-49-15-4B-D3-6E_Axo-Device-075_20250707_204037_000009_scored.edf
🔄 Processing: 04-49-15-4B-D3-6E_Axo-Device-075_20250708_205434_000010.edf
→ EEG: FpZ | EOG: EOG1 | EMG: EMG1


C:\Users\marin\anaconda3\envs\my_env_1\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 0.24.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


⚠️ EEGLAB .set export failed: For exporting to EEGLab to work, the module eeglabio is needed, but it could not be imported. Use the following installation method appropriate for your environment:

    pip install eeglabio
    conda install -c conda-forge eeglabio
✅ Done: 04-49-15-4B-D3-6E_Axo-Device-075_20250708_205434_000010_scored.edf

🔄 Processing: 11-E2-7B-2A-A0-71_Axo-Device-071_20250421_204304_000019.edf
→ EEG: FpZ | EOG: EOG1 | EMG: EMG1


C:\Users\marin\anaconda3\envs\my_env_1\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 0.24.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


⚠️ EEGLAB .set export failed: For exporting to EEGLab to work, the module eeglabio is needed, but it could not be imported. Use the following installation method appropriate for your environment:

    pip install eeglabio
    conda install -c conda-forge eeglabio
✅ Done: 11-E2-7B-2A-A0-71_Axo-Device-071_20250421_204304_000019_scored.edf

🔄 Processing: 11-E2-7B-2A-A0-71_Axo-Device-071_20250504_210352_000029.edf
→ EEG: FpZ | EOG: EOG1 | EMG: EMG1


...

⚠️ EEGLAB .set export failed: For exporting to EEGLab to work, the module eeglabio is needed, but it could not be imported. Use the following installation method appropriate for your environment:

    pip install eeglabio
    conda install -c conda-forge eeglabio
✅ Done: 33-54-23-13-A2-FD_Axo-Device-072_20250506_211332_000008_scored.edf

🔄 Processing: 33-54-23-13-A2-FD_Axo-Device-072_20250513_220301_000012_2.edf
→ EEG: FpZ | EOG: EOG1 | EMG: EMG1


C:\Users\marin\anaconda3\envs\my_env_1\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 0.24.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


⚠️ EEGLAB .set export failed: For exporting to EEGLab to work, the module eeglabio is needed, but it could not be imported. Use the following installation method appropriate for your environment:

    pip install eeglabio
    conda install -c conda-forge eeglabio
✅ Done: 33-54-23-13-A2-FD_Axo-Device-072_20250513_220301_000012_2_scored.edf

🔄 Processing: 33-54-23-13-A2-FD_Axo-Device-072_20250529_003439_000018.edf
→ EEG: FpZ | EOG: EOG1 | EMG: EMG1


C:\Users\marin\anaconda3\envs\my_env_1\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 0.24.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


⚠️ EEGLAB .set export failed: For exporting to EEGLab to work, the module eeglabio is needed, but it could not be imported. Use the following installation method appropriate for your environment:

    pip install eeglabio
    conda install -c conda-forge eeglabio
✅ Done: 33-54-23-13-A2-FD_Axo-Device-072_20250529_003439_000018_scored.edf

All done.


YASA Output and what it corresponds to in the manual scorings:

- Time in Bed (TIB): total duration of the hypnogram.  -> lights off - lights on
- Sleep Period Time (SPT): duration from first to last period of sleep. -> not present in manual scoring
- Wake After Sleep Onset (WASO): duration of wake periods within SPT. -> WASO
- Total Sleep Time (TST): total sleep duration in SPT. -> TST
- Sleep Onset Latency (SOL): Latency to first epoch of any sleep.  -> SleepLat
- SOL 5min: Latency to 5 minutes of persistent sleep (any stage).  -> not present in manual scoring
- REM latency: latency to first REM sleep.  -> REM latency
- Sleep Efficiency (SE): TST / TIB * 100 (%).  -> SleepEff
- Sleep Maintenance Efficiency (SME): TST / SPT * 100 (%).  -> not present in manual scoring
- Sleep Fragmentation Index: number of transitions from sleep to wake / hours of TST  -> not present in manual scoring
- Sleep stages amount and proportion of TST  -> for W, N1, N2, N3, REM: total and % of TST